# 04 — Segmentação (SAM) e atributos visuais

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/projeto-plumas/blob/main/04_sam_atributos/04_sam_atributos.ipynb)

**Objetivo:** usar as caixas do YOLO como prompt do SAM, **revisar** as máscaras e extrair atributos aparentes (em pixels).

**Escopo da professora:** seção 7 (máscara automática = proposta a revisar, não referência perfeita) e perguntas 2 e 3.

**Lê:** `03_yolo_treino/runs/.../best.pt` e `01_dados/frames/`.

**Grava:** `mascaras/`, `overlays/`, `deteccoes.csv`, `atributos.csv` (só máscaras aprovadas), `segmentacao_metricas.csv`.

**GPU:** sim (Ambiente de execução → Alterar tipo → GPU T4)


In [ ]:
# === Preparo (igual em todos os notebooks) ===
import os, sys, subprocess

REPO = "USUARIO/projeto-plumas"      # troque pelo seu (veja README)
REPO_DIR = "/content/projeto-plumas"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    try:                                   # repositório privado: secret GITHUB_TOKEN
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        _tok = None
    _url = f"https://{_tok}@github.com/{REPO}.git" if _tok else f"https://github.com/{REPO}.git"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True)
    else:
        _r = subprocess.run(["git", "clone", "-q", _url, REPO_DIR], capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("Falha no clone. Repositório privado? Configure o secret GITHUB_TOKEN (veja README).")
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:                                      # execução local: sobe até a raiz do repo
    _p = os.getcwd()
    while not os.path.exists(os.path.join(_p, "config.yaml")) and os.path.dirname(_p) != _p:
        _p = os.path.dirname(_p)
    os.chdir(_p)
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from pathlib import Path
from plumas.config import load_config, stage_dir, set_seed, log_experiment
cfg = load_config()
set_seed(cfg["seed"])
STAGE = "04_sam_atributos"
D = stage_dir(cfg, STAGE)                  # pasta desta etapa no Drive
print("Etapa:", STAGE, "| dados em:", D)


## 1. Detectar e segmentar
Para cada quadro, o YOLO gera caixas e o SAM gera **uma máscara por caixa** (a caixa é o prompt).
Processe a(s) sequência(s) inteira(s) em ordem: a etapa 05 precisa de todos os quadros, não só do teste.

Todas as medidas são **aparentes** (pixels ou proporção da imagem). Sem calibração da câmera, não há metros.

In [ ]:
import cv2, numpy as np, pandas as pd, torch
from ultralytics import YOLO, SAM
from plumas.features import mask_features

BEST = Path("")                  # caminho do best.pt, ex.: stage_dir(cfg,"03_yolo_treino")/"runs"/"C_yolo11n_img640"/"weights"/"best.pt"
SEQS = None                      # ex.: ["seq01"]; None = todas as sequências de frames.csv
CONF = cfg["yolo"]["conf"]

frames = pd.read_csv(stage_dir(cfg, "01_dados") / "frames.csv")
if SEQS:
    frames = frames[frames["sequence"].isin(SEQS)]
frames_dir = stage_dir(cfg, "01_dados") / "frames"
mask_dir = D / "mascaras"; ov_dir = D / "overlays"
mask_dir.mkdir(exist_ok=True); ov_dir.mkdir(exist_ok=True)

yolo = YOLO(str(BEST))
sam = SAM(cfg["sam"]["model"])
print("GPU:", torch.cuda.is_available(), "| quadros a processar:", len(frames))

In [ ]:
rows = []
for fr in frames["frame"]:
    img_path = frames_dir / fr
    img = cv2.imread(str(img_path)); H, W = img.shape[:2]
    det = yolo.predict(str(img_path), conf=CONF, iou=cfg["yolo"]["iou"], verbose=False)[0]
    boxes = det.boxes.xyxy.cpu().numpy().tolist()
    confs = det.boxes.conf.cpu().numpy().tolist()
    if not boxes:
        continue
    res = sam(str(img_path), bboxes=boxes, verbose=False)[0]      # caixa do YOLO como prompt
    if res.masks is None:
        continue
    masks = res.masks.data.cpu().numpy()
    overlay = img.copy()
    for k, (box, cf, m) in enumerate(zip(boxes, confs, masks)):
        m = (m > 0.5).astype(np.uint8)
        if m.shape != (H, W):
            m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
        mname = f"{Path(fr).stem}_d{k}.png"
        cv2.imwrite(str(mask_dir / mname), m * 255)
        feats = mask_features(m) or {}
        rows.append({"frame": fr, "det_id": k, "conf": cf, "x1": box[0], "y1": box[1], "x2": box[2], "y2": box[3],
                     "mask_file": mname, **feats})
        overlay[m > 0] = (0.6 * overlay[m > 0] + 0.4 * np.array([0, 0, 255])).astype(np.uint8)
        cv2.rectangle(overlay, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0, 255, 255), 2)
    cv2.imwrite(str(ov_dir / fr), overlay)

det_df = pd.DataFrame(rows)
det_df.to_csv(D / "deteccoes.csv", index=False)
print(len(det_df), "detecções em", det_df["frame"].nunique(), "quadros. Salvo em deteccoes.csv")

## 2. Revisar as máscaras (obrigatório)
Veja os overlays (máscara em vermelho, caixa em amarelo), principalmente onde a pluma é difusa ou de baixo contraste.
Anote em `REJEITADAS` os arquivos de máscara ruins (ex.: `"seq01_000120_d0.png"`). Só as aprovadas entram nos atributos.

In [ ]:
import matplotlib.pyplot as plt

def mostrar(pagina=0, por_pagina=12):
    sel = det_df.iloc[pagina * por_pagina:(pagina + 1) * por_pagina]
    fig, axs = plt.subplots(3, 4, figsize=(20, 11))
    for ax in axs.ravel():
        ax.axis("off")
    for ax, (_, r) in zip(axs.ravel(), sel.iterrows()):
        ax.imshow(cv2.cvtColor(cv2.imread(str(ov_dir / r["frame"])), cv2.COLOR_BGR2RGB))
        ax.set_title(r["mask_file"], fontsize=8)
    plt.tight_layout(); plt.show()

mostrar(0)      # troque o número da página para ver as outras

In [ ]:
REJEITADAS = []      # nomes de máscara rejeitadas
det_df["status"] = np.where(det_df["mask_file"].isin(REJEITADAS), "rejeitada", "ok")
det_df.to_csv(D / "deteccoes.csv", index=False)
att = det_df[(det_df["status"] == "ok") & det_df["area_px"].notna()]
att.to_csv(D / "atributos.csv", index=False)
print(det_df["status"].value_counts().to_dict(), "→ atributos.csv com", len(att), "linhas")
att[["frame", "area_px", "width_px", "height_px", "aspect_ratio", "cx", "cy", "orientation_deg", "frac_image"]].describe()

## 3. Qualidade da segmentação (pergunta 2)
Compare as máscaras **automáticas** com referências manuais em `mascaras_gt/<nome-do-quadro>.png` (30 a 50 quadros, preferindo casos difíceis).
A referência é a união das plumas do quadro (0/255). Isso mede o SAM, antes da sua revisão.

In [ ]:
from plumas.features import iou_dice

gt_dir = D / "mascaras_gt"
res_rows = []
for gt_path in sorted(gt_dir.glob("*.png")) if gt_dir.exists() else []:
    stem = gt_path.stem
    gt = cv2.imread(str(gt_path), 0) > 0
    pred = np.zeros_like(gt)
    for _, r in det_df[det_df["mask_file"].str.startswith(stem + "_d")].iterrows():
        pred |= cv2.imread(str(mask_dir / r["mask_file"]), 0) > 0
    res_rows.append({"frame": stem, **iou_dice(pred, gt)})

seg = pd.DataFrame(res_rows)
if len(seg):
    seg.to_csv(D / "segmentacao_metricas.csv", index=False)
    print(seg[["iou", "dice"]].describe().loc[["count", "mean", "std", "min", "max"]])
else:
    print("Sem máscaras de referência em", gt_dir, "(crie-as para responder à pergunta 2).")

In [ ]:
log_experiment(cfg, STAGE, {"pesos_yolo": str(BEST), "sam": cfg["sam"]["model"], "conf": CONF,
                            "n_deteccoes": int(len(det_df)), "n_rejeitadas": int((det_df["status"] == "rejeitada").sum()),
                            "iou_medio_vs_gt": float(seg["iou"].mean()) if len(seg) else None,
                            "decisao": "máscaras revisadas; atributos só das aprovadas"})